# Moon Temp — Sensor Offset Analysis

Cross-correlation analysis of ADS1115 channels (adc0, adc1, adc2) to determine
per-channel offsets that bring all sensors into agreement.

**Reference sensor:** adc0  
**Log file:** `moon_temp_logger/target/release/logs/2026-05-26.csv`

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 110

ModuleNotFoundError: No module named 'matplotlib'

## 1. Load Data

In [ ]:
CSV_PATH = "/Users/dek/claude_projects/moon_temp_logger/target/release/logs/2026-05-26.csv"

df = pd.read_csv(CSV_PATH, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Rows     : {len(df)}")
print(f"Span     : {df['timestamp'].iloc[0]}  →  {df['timestamp'].iloc[-1]}")
print(f"Duration : {df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]}")
df.head()

## 2. Raw Overview

In [ ]:
df[['adc0','adc1','adc2']].describe().round(1)

In [ ]:
fig, ax = plt.subplots()
for col, color in [('adc0','steelblue'), ('adc1','darkorange'), ('adc2','crimson')]:
    ax.plot(df['timestamp'], df[col], label=col, color=color, linewidth=0.9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title('Raw ADC readings — 2026-05-26')
ax.set_ylabel('ADC counts')
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3. Cross-Correlation Analysis

Cross-correlation finds the lag (in samples) at which two signals best align.
A lag of 0 means the sensors respond simultaneously.

In [ ]:
MAX_LAG = 20

def normalised_xcorr(x, y, max_lag=MAX_LAG):
    x = x - x.mean()
    y = y - y.mean()
    corr = np.correlate(x, y, mode='full')
    norm = np.sqrt((x**2).sum() * (y**2).sum())
    corr = corr / norm if norm else corr
    lags = np.arange(-len(x) + 1, len(x))
    mid  = len(lags) // 2
    return lags[mid - max_lag : mid + max_lag + 1], corr[mid - max_lag : mid + max_lag + 1]

def compute_offset(ref, sig, lag):
    n = len(ref)
    if lag >= 0: return round(float(np.mean(ref[lag:] - sig[:n - lag])))
    else:        return round(float(np.mean(ref[:n + lag] - sig[-lag:])))

def aligned_residuals(ref, sig_corr, lag):
    n = len(ref)
    if lag >= 0: return ref[lag:] - sig_corr[:n - lag]
    else:        return ref[:n + lag] - sig_corr[-lag:]

a0 = df['adc0'].values
a1 = df['adc1'].values
a2 = df['adc2'].values

lags1, xcorr1 = normalised_xcorr(a0, a1)
lags2, xcorr2 = normalised_xcorr(a0, a2)

lag1 = int(lags1[np.argmax(xcorr1)])
lag2 = int(lags2[np.argmax(xcorr2)])
r1   = xcorr1.max()
r2   = xcorr2.max()

off1 = compute_offset(a0, a1, lag1)
off2 = compute_offset(a0, a2, lag2)

res1 = aligned_residuals(a0, a1 + off1, lag1)
res2 = aligned_residuals(a0, a2 + off2, lag2)

print(f"adc1 : r={r1:.4f}  lag={lag1}  offset={off1:+d}  residual std={res1.std():.1f}")
print(f"adc2 : r={r2:.4f}  lag={lag2}  offset={off2:+d}  residual std={res2.std():.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, lags, xcorr, lag, r, label, color in [
    (axes[0], lags1, xcorr1, lag1, r1, 'adc1 vs adc0', 'darkorange'),
    (axes[1], lags2, xcorr2, lag2, r2, 'adc2 vs adc0', 'crimson'),
]:
    ax.bar(lags, xcorr, color=color, alpha=0.7, width=0.8)
    ax.axvline(lag, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{label}\npeak r={r:.4f}  lag={lag}')
    ax.set_xlabel('Lag (samples)')
    ax.set_ylabel('Normalised correlation')
    ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Corrected vs Reference

In [ ]:
corr = df.copy()
corr['adc1_corr'] = corr['adc1'] + off1
corr['adc2_corr'] = corr['adc2'] + off2

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for ax, pairs, title in [
    (axes[0], [('adc0','steelblue','adc0 (ref)'), ('adc1','darkorange','adc1 raw'), ('adc2','crimson','adc2 raw')], 'Before correction'),
    (axes[1], [('adc0','steelblue','adc0 (ref)'), ('adc1_corr','darkorange',f'adc1 {off1:+d}'), ('adc2_corr','crimson',f'adc2 {off2:+d}')], 'After correction'),
]:
    for col, color, lbl in pairs:
        ax.plot(corr['timestamp'], corr[col], label=lbl, color=color, linewidth=0.9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.set_title(f'2026-05-26 — {title}')
    ax.set_ylabel('ADC counts')
    ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, res, label, color in [
    (axes[0], res1, f'adc1 residual  std={res1.std():.1f}', 'darkorange'),
    (axes[1], res2, f'adc2 residual  std={res2.std():.1f}', 'crimson'),
]:
    ax.hist(res, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(label)
    ax.set_xlabel('Residual (counts)')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 5. Results Summary

In [ ]:
summary = pd.DataFrame({
    'channel':       ['adc0', 'adc1', 'adc2'],
    'role':          ['reference', 'corrected', 'corrected'],
    'offset_counts': [0, off1, off2],
    'offset_mV':     [0.0, round(off1 * 0.125, 3), round(off2 * 0.125, 3)],
    'xcorr_r':       [1.0, round(r1, 4), round(r2, 4)],
    'lag_samples':   [0, lag1, lag2],
    'residual_std':  ['-', f'{res1.std():.1f}', f'{res2.std():.1f}'],
})
print(summary.to_string(index=False))

## 6. MQTT Commands to Apply Offsets

Publish to topic `moon-temp-001/offset/cmd`:

In [ ]:
import json

for adc, offset in [(1, off1), (2, off2)]:
    payload = json.dumps({'adc': adc, 'offset': offset})
    print(f"Topic  : moon-temp-001/offset/cmd")
    print(f"Payload: {payload}")
    print()

---
# Part 2 — Verification (2026-05-27)

Offsets applied to device on **2026-05-26** via MQTT:
- `{"adc": 1, "offset": -63}`
- `{"adc": 2, "offset": -71}`

Run the cells below after the 2026-05-27 log has been collected to verify the offsets are working.
Expected outcome: all three channels track closely with no systematic offset remaining.

In [ ]:
CSV_27 = "/Users/dek/claude_projects/moon_temp_logger/target/release/logs/2026-05-27.csv"

df27 = pd.read_csv(CSV_27, parse_dates=['timestamp'])
df27 = df27.sort_values('timestamp').reset_index(drop=True)

print(f"Rows     : {len(df27)}")
print(f"Span     : {df27['timestamp'].iloc[0]}  →  {df27['timestamp'].iloc[-1]}")
print(f"Duration : {df27['timestamp'].iloc[-1] - df27['timestamp'].iloc[0]}")
df27.head()

In [ ]:
df27[['adc0','adc1','adc2']].describe().round(1)

In [ ]:
# Raw readings — offsets now baked in on the device side
fig, ax = plt.subplots()
for col, color in [('adc0','steelblue'), ('adc1','darkorange'), ('adc2','crimson')]:
    ax.plot(df27['timestamp'], df27[col], label=col, color=color, linewidth=0.9)

heat_start = pd.Timestamp('2026-05-27 09:51:00')
heat_end   = pd.Timestamp('2026-05-27 09:58:15')
ax.axvspan(heat_start, heat_end, color='gold', alpha=0.25, label='heat test')
ax.text(
    heat_start, ax.get_ylim()[1],
    ' heat test\n ADC0→1→2',
    fontsize=8, color='goldenrod', va='top',
)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title('2026-05-27 — Raw ADC readings (post-offset)')
ax.set_ylabel('ADC counts')
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Inter-channel residuals — should be centred near zero with low std
v0 = df27['adc0'].values
v1 = df27['adc1'].values
v2 = df27['adc2'].values

diff1 = v0 - v1
diff2 = v0 - v2

print(f"adc0 - adc1 : mean={diff1.mean():+.1f}  std={diff1.std():.1f} counts  ({diff1.std()*0.125:.2f} mV)")
print(f"adc0 - adc2 : mean={diff2.mean():+.1f}  std={diff2.std():.1f} counts  ({diff2.std()*0.125:.2f} mV)")

In [ ]:
# Difference over time — flat line near zero = offsets working correctly
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, diff, label, color in [
    (axes[0], diff1, 'adc0 − adc1', 'darkorange'),
    (axes[1], diff2, 'adc0 − adc2', 'crimson'),
]:
    ax.plot(df27['timestamp'], diff, color=color, linewidth=0.8, alpha=0.8)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.set_title(f'{label}  (mean={np.mean(diff):+.1f}  std={np.std(diff):.1f})')
    ax.set_ylabel('Δ counts')

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Residual histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, diff, label, color in [
    (axes[0], diff1, f'adc0 − adc1  std={diff1.std():.1f}', 'darkorange'),
    (axes[1], diff2, f'adc0 − adc2  std={diff2.std():.1f}', 'crimson'),
]:
    ax.hist(diff, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'2026-05-27 — {label}')
    ax.set_xlabel('Δ counts')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

---
# Part 3 — Heat Response Test (2026-05-27 ~09:51–09:58 MDT)

Sensor liveness check: hand firmly gripped each sensor for ~1 minute in sequence (ADC0 → ADC1 → ADC2).
Collection interval changed to 1 second for this window.

NTC thermistors respond to heat with a **decrease** in resistance → lower ADC counts.

In [ ]:
TEST_START = '2026-05-27 09:51:00'
TEST_END   = '2026-05-27 09:59:00'

df_full = pd.read_csv(
    "/Users/dek/claude_projects/moon_temp_logger/target/release/logs/2026-05-27.csv",
    parse_dates=['timestamp']
)
df_full = df_full.sort_values('timestamp').reset_index(drop=True)

test = df_full[(df_full['timestamp'] >= TEST_START) & (df_full['timestamp'] <= TEST_END)].copy()

print(f"Rows : {len(test)}")
print(f"Span : {test['timestamp'].iloc[0]}  →  {test['timestamp'].iloc[-1]}")
print(f"Sample interval (median): {test['timestamp'].diff().dt.total_seconds().median():.0f}s")
print()
for col in ['adc0','adc1','adc2']:
    drop = test[col].max() - test[col].min()
    print(f"{col}: baseline={test[col].iloc[0]}  min={test[col].min()}  drop={drop} counts ({drop*0.125:.1f} mV)")
test.head()

In [ ]:
# Per-channel subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
colors = {'adc0': 'steelblue', 'adc1': 'darkorange', 'adc2': 'crimson'}
labels = {'adc0': 'ADC0 (hand first)', 'adc1': 'ADC1 (hand second)', 'adc2': 'ADC2 (hand third)'}

for ax, col in zip(axes, ['adc0', 'adc1', 'adc2']):
    ax.plot(test['timestamp'], test[col], color=colors[col], linewidth=1.0, label=labels[col])
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax.set_ylabel('ADC counts')
    ax.legend(loc='upper left')
    ax.set_title(labels[col])

axes[-1].set_xlabel('Time (MDT)')
fig.suptitle('Heat Response Test — 2026-05-27 ~09:51–09:58 MDT', fontsize=13)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Overlay — sequential drops clearly visible
fig, ax = plt.subplots(figsize=(14, 5))
for col in ['adc0', 'adc1', 'adc2']:
    ax.plot(test['timestamp'], test[col], color=colors[col], linewidth=1.0, label=labels[col], alpha=0.85)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_ylabel('ADC counts')
ax.set_title('Heat Response Test — all channels overlay')
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

---
# Part 4 — Progression: May 26 → May 27 → May 28

Compare inter-channel offsets across all three days to see whether the applied corrections helped, hurt, or had no effect.

In [ ]:
CSV_28 = "/Users/dek/claude_projects/moon_temp_logger/target/release/logs/2026-05-28.csv"

df28 = pd.read_csv(CSV_28, parse_dates=['timestamp'])
df28 = df28.sort_values('timestamp').reset_index(drop=True)

print(f"Rows     : {len(df28)}")
print(f"Span     : {df28['timestamp'].iloc[0]}  →  {df28['timestamp'].iloc[-1]}")
print(f"Duration : {df28['timestamp'].iloc[-1] - df28['timestamp'].iloc[0]}")
df28.head()

In [ ]:
# Inter-channel difference summary across all three days
# Positive mean = adc0 reads higher than the other channel

rows = []
for label, dframe in [('2026-05-26 (baseline)', df), ('2026-05-27 (post-offset)', df27), ('2026-05-28 (current)', df28)]:
    for pair, ch in [('adc0 − adc1', 'adc1'), ('adc0 − adc2', 'adc2')]:
        d = dframe['adc0'] - dframe[ch]
        rows.append({'date': label, 'pair': pair, 'mean': round(d.mean(), 1), 'std': round(d.std(), 1), 'median': round(d.median(), 1)})

progression = pd.DataFrame(rows)
print(progression.to_string(index=False))

In [ ]:
# Raw readings for May 28
fig, ax = plt.subplots()
for col, color in [('adc0','steelblue'), ('adc1','darkorange'), ('adc2','crimson')]:
    ax.plot(df28['timestamp'], df28[col], label=col, color=color, linewidth=0.9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title('Raw ADC readings — 2026-05-28')
ax.set_ylabel('ADC counts')
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# adc0 − adc1 difference over time for all three days (same scale)
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharey=True)

for ax, label, dframe, color in [
    (axes[0], '2026-05-26 (baseline, no correction)', df,   'steelblue'),
    (axes[1], '2026-05-27 (offset -63 applied)',      df27, 'darkorange'),
    (axes[2], '2026-05-28 (current)',                 df28, 'crimson'),
]:
    diff = dframe['adc0'] - dframe['adc1']
    ax.plot(dframe['timestamp'], diff, color=color, linewidth=0.8, alpha=0.85)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.set_title(f'adc0 − adc1  |  {label}  (mean={diff.mean():+.1f}  median={diff.median():+.0f})')
    ax.set_ylabel('Δ counts')

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Recommended offset to send for May 28 correction
median_gap_adc1 = int((df28['adc0'] - df28['adc1']).median())
median_gap_adc2 = int((df28['adc0'] - df28['adc2']).median())

print("=== Recommended MQTT offset commands (based on May 28 median gap) ===")
print()
for adc, gap in [(1, median_gap_adc1), (2, median_gap_adc2)]:
    payload = json.dumps({'adc': adc, 'offset': gap})
    print(f"Topic  : moon-temp-001/offset/cmd")
    print(f"Payload: {payload}")
    print(f"  (adc{adc} is currently reading {gap:+d} counts {'below' if gap > 0 else 'above'} adc0)")
    print()

---
# Part 5 — Temperature Conversion & Multi-Day Summary

**Sensor:** MF58 10 kΩ NTC, B = 3950 K  
**Circuit:** VCC (3.3 V) → 10 kΩ fixed → ADS1115 input → NTC → GND  
**ADC LSB:** 0.125 mV (PGA ±4.096 V, 16-bit)

<img src="Assets/SensorGeneralConstruction.png" width="480" alt="Sensor construction — NTC thermistor thermal-epoxied to 4″×1.5″×¼″ cold steel bar"/>

### Hardware Reference

**Sensors, ESP32-C3, and power supply — bench setup**

<img src="Assets/Moon_Temp_SensorCalibration_8636.jpeg" width="600" alt="Three NTC sensor assemblies on wooden mount, ESP32-C3 with antenna, and USB power supply"/>

---

**Static chamber — 8 ft deep concrete pit (looking down)**

<img src="Assets/Moon_Temp_SensorCalibration_8637.jpeg" width="600" alt="8-foot deep concrete block pit used as static thermal chamber"/>

---

**Inside the pit — sensor deployment at depth**

<img src="Assets/Moon_Temp_SensorCalibration_8638.jpeg" width="600" alt="Bottom of pit showing steel bar sensor assemblies and ESP32 electronics enclosure deployed in-situ"/>

In [ ]:
VCC      = 3.3
R_SERIES = 10_000
R0       = 10_000
B        = 3950
T0_K     = 298.15  # 25°C reference

def counts_to_celsius(counts):
    V = counts * 0.000125
    R = V * R_SERIES / (VCC - V)
    return 1 / (1/T0_K + (1/B) * np.log(R / R0)) - 273.15

# Load May 29
CSV_29 = "/Users/dek/claude_projects/moon_temp_logger/target/release/logs/2026-05-29.csv"
df29 = pd.read_csv(CSV_29, parse_dates=['timestamp'])
df29 = df29.sort_values('timestamp').reset_index(drop=True)
print(f"May 29 — Rows: {len(df29)}  Span: {df29['timestamp'].iloc[0]}  →  {df29['timestamp'].iloc[-1]}")

In [ ]:
# Apply temperature conversion to all days
for label, dframe in [('26', df), ('27', df27), ('28', df28), ('29', df29)]:
    for ch in ['adc0', 'adc1', 'adc2']:
        dframe[ch.replace('adc', 'temp')] = counts_to_celsius(dframe[ch])

# Per-channel summary for May 29
print("=" * 60)
print("Temperature summary — May 29 (°C)")
print("=" * 60)
print(df29[['temp0','temp1','temp2']].describe().round(3).to_string())
print()
for pair, a, b in [('temp0 − temp1','temp0','temp1'), ('temp0 − temp2','temp0','temp2'), ('temp1 − temp2','temp1','temp2')]:
    d = df29[a] - df29[b]
    print(f"{pair} : mean={d.mean():+.3f}°C  std={d.std():.3f}°C")
print()
lsb_deg = abs(counts_to_celsius(12784) - counts_to_celsius(12785))
print(f"1 LSB (1 count) ≈ {lsb_deg*1000:.2f} m°C  ({lsb_deg:.4f}°C)")

In [ ]:
# Temperature plot — May 29
fig, ax = plt.subplots()
for col, color, label in [('temp0','steelblue','ch0'), ('temp1','darkorange','ch1'), ('temp2','crimson','ch2')]:
    ax.plot(df29['timestamp'], df29[col], color=color, linewidth=0.9, label=label)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title('Temperature (°C) — 2026-05-29  [MF58 10K NTC, B=3950]')
ax.set_ylabel('°C')
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Multi-day mean temperature progression (channels averaged)
CSV_30 = "/Users/dek/claude_projects/moon_temp_logger/target/release/logs/2026-05-30.csv"
df30 = pd.read_csv(CSV_30, parse_dates=['timestamp'])
df30 = df30.sort_values('timestamp').reset_index(drop=True)
for ch in ['adc0', 'adc1', 'adc2']:
    df30[ch.replace('adc', 'temp')] = counts_to_celsius(df30[ch])

rows = []
for label, dframe in [
    ('May 26\n(baseline)', df),
    ('May 27\n(post-offset)', df27),
    ('May 28\n(current)', df28),
    ('May 29\n(corrected)', df29),
    ('May 30\n(stable)', df30),
]:
    if '27' in label:
        d = dframe[~((dframe['timestamp'] >= '2026-05-27 09:51:00') & (dframe['timestamp'] <= '2026-05-27 09:58:15'))]
    else:
        d = dframe
    rows.append({
        'day': label,
        'ch0_mean': d['temp0'].mean(),
        'ch1_mean': d['temp1'].mean(),
        'ch2_mean': d['temp2'].mean(),
        'ch0_std':  d['temp0'].std(),
        'ch1_std':  d['temp1'].std(),
        'ch2_std':  d['temp2'].std(),
        'spread':   max(abs(d['temp0']-d['temp1']).mean(),
                        abs(d['temp0']-d['temp2']).mean(),
                        abs(d['temp1']-d['temp2']).mean()),
    })

prog_temp = pd.DataFrame(rows).set_index('day')
print("Multi-day temperature summary (°C, heat test excluded from May 27)")
print()
print(prog_temp.round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(rows))
for col, color, label in [('ch0_mean','steelblue','ch0'), ('ch1_mean','darkorange','ch1'), ('ch2_mean','crimson','ch2')]:
    means = prog_temp[col].values
    stds  = prog_temp[col.replace('mean','std')].values
    ax.errorbar(x, means, yerr=stds, fmt='o-', color=color, label=label, capsize=4, linewidth=1.5)
ax.set_xticks(list(x))
ax.set_xticklabels(prog_temp.index, fontsize=9)
ax.set_ylabel('°C')
ax.set_title('Mean temperature per day — all channels (± 1 std)')
ax.legend()
plt.tight_layout()
plt.show()

---
# Part 6 — May 30 Analysis

777 readings, 00:00–12:56. Pit warmed ~0.22°C overnight vs May 29 (counts dropped 12784 → 12720). Inter-sensor spread < 0.06°C.

In [ ]:
print("=" * 60)
print("Temperature summary — May 30 (°C)")
print("=" * 60)
print(df30[['temp0','temp1','temp2']].describe().round(3).to_string())
print()
for pair, a, b in [('temp0 − temp1','temp0','temp1'), ('temp0 − temp2','temp0','temp2'), ('temp1 − temp2','temp1','temp2')]:
    d = df30[a] - df30[b]
    print(f"{pair} : mean={d.mean():+.3f}°C  median={d.median():+.3f}°C  std={d.std():.3f}°C")
print()
print(f"First: {df30['timestamp'].iloc[0]}  t0={df30['temp0'].iloc[0]:.3f}  t1={df30['temp1'].iloc[0]:.3f}  t2={df30['temp2'].iloc[0]:.3f}")
print(f"Last : {df30['timestamp'].iloc[-1]}  t0={df30['temp0'].iloc[-1]:.3f}  t1={df30['temp1'].iloc[-1]:.3f}  t2={df30['temp2'].iloc[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots()
for col, color, label in [('temp0','steelblue','ch0'), ('temp1','darkorange','ch1'), ('temp2','crimson','ch2')]:
    ax.plot(df30['timestamp'], df30[col], color=color, linewidth=0.9, label=label)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title('Temperature (°C) — 2026-05-30  [MF58 10K NTC, B=3950]')
ax.set_ylabel('°C')
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()